# Case-study NB2 — Full Experiment & Verification

Runs LocalOnly, FedProto-64, FedProto-8, and NEST (64↔8) on the frozen case-study graph generated by NB1. Also verifies embedding alignment (cosine similarity).

In [ ]:
# 1. Imports and dynamic function loading from source notebooks
from pathlib import Path
import os, glob, json, time, random, copy
import numpy as np
import pandas as pd
import torch
import nbformat
import torch.nn.functional as F
from sklearn.metrics.pairwise import cosine_similarity

# --- If PyTorch Geometric is not installed, uncomment the line below ---
# !pip install torch-geometric

SEED = 42
IN_DIR = Path('/kaggle/input/peeling_chain_case_study') # Kaggle dataset path if uploaded, else fallback
OUT = Path('/kaggle/working/case_study_results')
OUT.mkdir(parents=True, exist_ok=True)

# Try local working dir first if run consecutively, else look in Kaggle inputs
FROZEN_PATH = Path('/kaggle/working/peeling_chain_case_study/frozen_case_graph.pt')
if not FROZEN_PATH.exists():
    hits = list(Path('/kaggle/input').rglob('frozen_case_graph.pt'))
    if hits: FROZEN_PATH = hits[0]
    else: raise FileNotFoundError("Cannot find frozen_case_graph.pt")

print(f"Using frozen graph at: {FROZEN_PATH}")

def find_unique(filename):
    hits = sorted(Path(p) for p in glob.glob('/kaggle/input/**/' + filename, recursive=True))
    if not hits: raise FileNotFoundError(f'Missing required input: {filename}')
    return hits[0]

def source_of_marker(nb_path, marker):
    nb = nbformat.read(nb_path, as_version=4)
    matches = [''.join(cell.source) for cell in nb.cells if cell.cell_type == 'code' and marker in ''.join(cell.source)]
    if len(matches) != 1: raise RuntimeError(f'Expected exactly one source cell for {marker!r}; found {len(matches)}')
    return matches[0]

N1 = find_unique('notebook1.ipynb') if not Path('notebook1.ipynb').exists() else Path('notebook1.ipynb')
try: N2 = find_unique('notebook2.ipynb')
except FileNotFoundError: N2 = find_unique('nest_rank_aware_aggregation.ipynb')

MARKERS = [
    (N1, 'class ExperimentConfig'),
    (N2, 'class SAGEGATEncoder'),
    (N1, 'class FullGAT'),
    (N2, 'def evaluate_tuned'),
    (N2, 'def ssl_pretrain_client'),
    (N1, 'def compute_embedding_prototypes'),
    (N2, 'DEVICE_TIER_DIMS = (8, 16, 32, 64)'),
    (N2, 'def rank_aware_contrib_agg'),
    (N1, 'def supervised_round_client'),
    (N2, 'def supervised_round_client_ep'),
    (N1, 'def fedavg_state_selective'),
    (N1, 'def fedper_head_finetune'),
    (N1, 'def run_pgfcl')
]

for nb, marker in MARKERS:
    try:
        exec(compile(source_of_marker(nb, marker), f'<{nb.name}:{marker}>', 'exec'), globals())
    except Exception as e:
        print(f"Warning: Marker {marker} failed to load directly: {e}")
print("Source functions loaded successfully.")

# Edge index helpers (copied verbatim to bypass the full-dataset split logic in NB1)
def get_local_edge_index(data, mask, device):
    ei = data.edge_index.to(device); m = mask.to(device); keep = m[ei[0]] & m[ei[1]]
    return ei[:, keep]

def get_inductive_edge_index(data, train_mask, test_mask, device):
    ei = data.edge_index.to(device); tr = train_mask.to(device); te = test_mask.to(device); all_m = tr | te
    keep = (te[ei[1]] & all_m[ei[0]]) | (te[ei[0]] & te[ei[1]])
    return ei[:, keep]

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)


In [ ]:
# 2. Load the frozen topology (STRICTLY ON CPU)
# PyG expects the graph to live on the CPU, and local training functions push to GPU dynamically.
frozen = torch.load(FROZEN_PATH, weights_only=False, map_location='cpu')
case = frozen['data']
clients = frozen['clients']

# Reconstruct mask tensors if they were flattened (keep them on CPU)
for c in clients:
    if 'train_mask_indices' in c and 'train_mask' not in c:
        m = torch.zeros(case.num_nodes, dtype=torch.bool)
        m[c['train_mask_indices']] = True
        c['train_mask'] = m
    if 'test_mask_indices' in c and 'test_mask' not in c:
        m = torch.zeros(case.num_nodes, dtype=torch.bool)
        m[c['test_mask_indices']] = True
        c['test_mask'] = m

print(f"Loaded frozen graph: {case.num_nodes} nodes, {case.num_edges} edges")


In [ ]:
# 3. Custom runners that return the trained global model for embedding extraction
# We rewrite the main loop cleanly to retain access to `global_model` and `z`.
# data is kept on CPU here; models are pushed to DEVICE.

def run_case_local_only(data, client, cfg):
    model = FullGAT(data.num_node_features, cfg).to(DEVICE)
    best_f1, best_m = 0.0, None
    for r in range(cfg.global_rounds):
        supervised_round_client(model, data, client, DEVICE, cfg, None, None, 0.0, None)
        m = evaluate_tuned(model, data, client['train_mask'], client['test_mask'], DEVICE, cfg)
        if m['f1'] > best_f1: best_f1, best_m = m['f1'], copy.deepcopy(m)
    return best_m, model

def run_case_fedproto(data, clients, cfg, proto_dim):
    global_model = FullGAT(data.num_node_features, cfg).to(DEVICE)
    global_model = ssl_pretrain_phase(global_model, data, clients, DEVICE, cfg, verbose=False)
    
    sizes = [c['n_train'] for c in clients]
    best_f1, best_m, best_model_state = 0.0, None, None
    
    for r in range(cfg.global_rounds):
        lam = cfg.lam_max * min(1.0, r / max(cfg.lam_warmup_rounds, 1))
        local_models, client_protos = [], []
        
        for c in clients:
            with torch.no_grad(): z = global_model.encoder(data.x.to(DEVICE), data.edge_index.to(DEVICE))
            cp, _ = compute_embedding_prototypes(z, data, c['train_mask'], DEVICE, cfg)
            client_protos.append({cls: p[:proto_dim] for cls, p in cp.items()})
            
            lm = FullGAT(data.num_node_features, cfg).to(DEVICE)
            lm.load_state_dict(global_model.state_dict())
            local_models.append(lm)
            
        global_protos = {cls: torch.zeros(proto_dim, device=DEVICE) for cls in [0, 1]}
        for cls in [0, 1]:
            for i in range(len(clients)):
                global_protos[cls] += client_protos[i][cls] * (sizes[i] / sum(sizes))
                
        for i, c in enumerate(clients):
            supervised_round_client(local_models[i], data, c, DEVICE, cfg, global_model, global_protos, lam, proto_dim)
            
        global_model = fedavg_state_selective(global_model, local_models, sizes)
        
        m = evaluate_tuned(global_model, data, clients[1]['train_mask'], clients[1]['test_mask'], DEVICE, cfg, proto_dim=proto_dim)
        if m['f1'] > best_f1: 
            best_f1, best_m = m['f1'], copy.deepcopy(m)
            best_model_state = copy.deepcopy(global_model.state_dict())
            
    global_model.load_state_dict(best_model_state)
    return best_m, global_model

def run_case_nest(data, clients, cfg, tier_map):
    global_model = FullGAT(data.num_node_features, cfg).to(DEVICE)
    global_model = ssl_pretrain_phase(global_model, data, clients, DEVICE, cfg, verbose=False)
    
    sizes = [c['n_train'] for c in clients]
    tier_dims = (8, 64)
    client_tiers = [tier_map[c['id']] for c in clients]
    prev_protos = None
    best_f1, best_m, best_model_state = 0.0, None, None
    
    for r in range(cfg.global_rounds):
        lam = cfg.lam_max * min(1.0, r / max(cfg.lam_warmup_rounds, 1))
        local_models, client_full_protos = [], []
        
        for c in clients:
            with torch.no_grad(): z = global_model.encoder(data.x.to(DEVICE), data.edge_index.to(DEVICE))
            full_p, _ = compute_embedding_prototypes(z, data, c['train_mask'], DEVICE, cfg)
            client_full_protos.append(full_p)
            
            lm = FullGAT(data.num_node_features, cfg).to(DEVICE)
            lm.load_state_dict(global_model.state_dict())
            local_models.append(lm)
            
        global_protos = rank_aware_contrib_agg(client_full_protos, client_tiers, sizes, prev_protos, cfg, tier_dims)
        prev_protos = {cls: p.detach().clone() for cls, p in global_protos.items()}
        
        for i, c in enumerate(clients):
            supervised_round_client_ep(local_models[i], data, c, DEVICE, cfg, None, global_protos, lam, client_tiers[i], tier_dims)
            
        global_model = fedavg_state_selective(global_model, local_models, sizes)
        
        m = evaluate_tuned(global_model, data, clients[1]['train_mask'], clients[1]['test_mask'], DEVICE, cfg, proto_dim=None)
        if m['f1'] > best_f1: 
            best_f1, best_m = m['f1'], copy.deepcopy(m)
            best_model_state = copy.deepcopy(global_model.state_dict())
            
    global_model.load_state_dict(best_model_state)
    return best_m, global_model


In [ ]:
# 4. Execute the suite
SMOKE_TEST = True # Toggle to False for full 5-seed run
SEEDS = [42] if SMOKE_TEST else [42, 43, 44, 45, 46]

print(f"Starting execution. SMOKE_TEST={SMOKE_TEST}")
cfg = ExperimentConfig(
    global_rounds=5 if SMOKE_TEST else 100,
    ssl_pretrain_rounds=1 if SMOKE_TEST else 20,
    ssl_epochs=20,
    sup_epochs=25,
    head_finetune_rounds=1 if SMOKE_TEST else 5,
    use_saliency=False, use_calibration=False,
    n_clients=2
)

all_results = {}
final_models = {}

for s in SEEDS:
    print(f"\n--- SEED {s} ---")
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)
    
    seed_res = {}
        
    print("Running LocalOnly B...")
    res_loc, mod_loc = run_case_local_only(case, clients[1], cfg)
    seed_res['LocalOnly_B'] = res_loc
    
    print("Running FedProto-64...")
    res_fp64, mod_fp64 = run_case_fedproto(case, clients, cfg, proto_dim=64)
    seed_res['FedProto_64'] = res_fp64
    
    print("Running FedProto-8...")
    res_fp8, mod_fp8 = run_case_fedproto(case, clients, cfg, proto_dim=8)
    seed_res['FedProto_8'] = res_fp8
    
    print("Running NEST (64↔8)...")
    res_nest, mod_nest = run_case_nest(case, clients, cfg, tier_map={0: 64, 1: 8})
    seed_res['NEST_64_8'] = res_nest
    
    all_results[s] = seed_res
    if s == 42: # Retain one set of models for embedding analysis
        final_models = {'FedProto_8': mod_fp8, 'NEST_64_8': mod_nest}
        
    print(f"Seed {s} Client B Recall:")
    print(f"  Local B:   {res_loc['rec']:.3f}")
    print(f"  FP-64:     {res_fp64['rec']:.3f}")
    print(f"  FP-8:      {res_fp8['rec']:.3f}")
    print(f"  NEST 64↔8: {res_nest['rec']:.3f}")

with open(OUT / 'case_study_metrics.json', 'w') as f:
    json.dump(all_results, f, indent=2)


In [ ]:
# 5. Embedding-Level Verification
# Check that Client B's trained 8-D representation actually aligns with Client A's first 8 dimensions.

print("\n--- Embedding Space Verification ---")
mod_nest = final_models['NEST_64_8']
mod_nest.eval()

with torch.no_grad():
    z_nest = mod_nest.encoder(case.x.to(DEVICE), case.edge_index.to(DEVICE))
    
# Extract embeddings for nodes in A and nodes in B
a_nodes = torch.where(clients[0]['train_mask'] | clients[0]['test_mask'])[0].to(DEVICE)
b_nodes = torch.where(clients[1]['train_mask'] | clients[1]['test_mask'])[0].to(DEVICE)

z_a = z_nest[a_nodes] # [N_A, 64]
z_b = z_nest[b_nodes] # [N_B, 64]

# In NEST, Client B only supervised its first 8 dims, while Client A supervised all 64 dims.
# Do A and B occupy a coherent shared space in the first 8 dimensions?
sim_matrix_8 = cosine_similarity(z_a[:, :8].cpu().numpy(), z_b[:, :8].cpu().numpy())
mean_sim_8 = sim_matrix_8.mean()

print(f"Mean Cosine Similarity between A and B in shared 8-D subspace: {mean_sim_8:.4f}")

# Sanity check: Do they align in the unshared dimensions? (They shouldn't, as B didn't train them).
sim_matrix_unshared = cosine_similarity(z_a[:, 8:].cpu().numpy(), z_b[:, 8:].cpu().numpy())
mean_sim_unshared = sim_matrix_unshared.mean()
print(f"Mean Cosine Similarity between A and B in unshared 56-D subspace: {mean_sim_unshared:.4f}")

if mean_sim_8 > mean_sim_unshared + 0.1:
    print("✅ Verification passed: The nested 8-D space exhibits alignment, whereas unshared dimensions do not.")
else:
    print("⚠️ Warning: The 8-D subspace does not show significantly higher alignment than untrained dimensions.")
